In [2]:
with open('Panchatantra.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

In [6]:
import re
words = re.findall(r'\w+', text)
print(words)

['panchatantra', 'pandit', 'vishnu', 'sharma', 'translated', 'by', 'g', 'l', 'chandiramani', 'copyright', 'sheila', 'g', 'chandiramani', 'first', 'in', 'rupa', 'paperback', '1991', 'twentieth', 'impression', '2011', 'published', 'by', 'rupa', 'publications', 'india', 'pvt', 'ltd', '7', '16', 'ansari', 'road', 'daryaganj', 'new', 'delhi', '110', '002', 'sales', 'centres', 'allahabad', 'bengaluru', 'chennai', 'hyderabad', 'jaipur', 'kathmandu', 'kolkata', 'mumbai', 'all', 'rights', 'reserved', 'no', 'part', 'of', 'this', 'publication', 'may', 'be', 'reproduced', 'stored', 'in', 'a', 'retrieval', 'system', 'or', 'transmitted', 'in', 'any', 'form', 'or', 'by', 'any', 'means', 'electronic', 'mechanical', 'photocopying', 'recording', 'or', 'otherwise', 'without', 'the', 'prior', 'permission', 'of', 'the', 'publishers', 'typeset', 'by', 'mindways', 'design', '1410', 'chiranjiv', 'tower', '43', 'nehru', 'place', 'new', 'delhi', '110', '019', 'printed', 'in', 'india', 'by', 'gopsons', 'papers',

In [7]:
len(words)

70110

In [5]:
unikdict = sorted(set(words))
word_to_idx = {word: idx for idx, word in enumerate(unikdict)}
idx_to_word = {idx: word for idx, word in enumerate(unikdict)}
unikdict_size = len(unikdict)
print(unikdict_size)

6442


In [11]:
sequence_length = 5
sequences = []
next_words = []

for i in range(len(words) - sequence_length):
    seq = words[i:i + sequence_length]  
    target = words[i + sequence_length]
    sequences.append([word_to_idx[word] for word in seq])  
    next_words.append(word_to_idx[target])

sequences

[[4050, 4059, 6117, 5040, 5817],
 [4059, 6117, 5040, 5817, 973],
 [6117, 5040, 5817, 973, 2409],
 [5040, 5817, 973, 2409, 3225],
 [5817, 973, 2409, 3225, 1075],
 [973, 2409, 3225, 1075, 1315],
 [2409, 3225, 1075, 1315, 5055],
 [3225, 1075, 1315, 5055, 2409],
 [1075, 1315, 5055, 2409, 1075],
 [1315, 5055, 2409, 1075, 2224],
 [5055, 2409, 1075, 2224, 2937],
 [2409, 1075, 2224, 2937, 4820],
 [1075, 2224, 2937, 4820, 4066],
 [2224, 2937, 4820, 4066, 69],
 [2937, 4820, 4066, 69, 5897],
 [4820, 4066, 69, 5897, 2927],
 [4066, 69, 5897, 2927, 94],
 [69, 5897, 2927, 94, 4399],
 [5897, 2927, 94, 4399, 973],
 [2927, 94, 4399, 973, 4820],
 [94, 4399, 973, 4820, 4398],
 [4399, 973, 4820, 4398, 2956],
 [973, 4820, 4398, 2956, 4421],
 [4820, 4398, 2956, 4421, 3423],
 [4398, 2956, 4421, 3423, 204],
 [2956, 4421, 3423, 204, 49],
 [4421, 3423, 204, 49, 483],
 [3423, 204, 49, 483, 4763],
 [204, 49, 483, 4763, 1459],
 [49, 483, 4763, 1459, 3799],
 [483, 4763, 1459, 3799, 1522],
 [4763, 1459, 3799, 1522, 2

In [12]:
import torch
X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(next_words, dtype=torch.long)

In [ ]:
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # Word embeddings
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)  # RNN layer
        self.fc = nn.Linear(hidden_dim, vocab_size)  # Output layer
    
    def forward(self, x, hidden):
        # x shape: (batch_size, sequence_length)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        output, hidden = self.rnn(embedded, hidden)  # output: (batch_size, seq_len, hidden_dim)
        output = self.fc(output[:, -1, :])  # Take last time step: (batch_size, vocab_size)
        return output, hidden
    
    def init_hidden(self, batch_size):
        # Initialize hidden state with zeros
        return torch.zeros(1, batch_size, hidden_dim)

# Hyperparameters
embedding_dim = 100  # Size of word embeddings
hidden_dim = 256     # Size of RNN hidden state
model = SimpleRNN(unikdict_size, embedding_dim, hidden_dim)